# Interfaz básica de carrito de compra

Esta interfaz sigue el patrón del ejemplo de minimarket: datos, funciones, componentes, conexión de botones y visualización.

Ejecuta las celdas en orden en Google Colab.

In [ ]:
# Celda 1. Base de datos de productos
import pandas as pd

datos = [
    {'codigo': 'A001', 'nombre': 'Arroz 1 kg', 'categoria': 'Abarrotes', 'precio': 1500, 'stock': 20},
    {'codigo': 'A002', 'nombre': 'Fideos 400 g', 'categoria': 'Abarrotes', 'precio': 900, 'stock': 35},
    {'codigo': 'B001', 'nombre': 'Leche entera 1 L', 'categoria': 'Lácteos', 'precio': 1200, 'stock': 18},
    {'codigo': 'B002', 'nombre': 'Yogur natural', 'categoria': 'Lácteos', 'precio': 850, 'stock': 25},
    {'codigo': 'C001', 'nombre': 'Bebida cola 1.5 L', 'categoria': 'Bebidas', 'precio': 1900, 'stock': 12},
    {'codigo': 'C002', 'nombre': 'Agua mineral 1.5 L', 'categoria': 'Bebidas', 'precio': 1000, 'stock': 30},
    {'codigo': 'D001', 'nombre': 'Jabón líquido', 'categoria': 'Limpieza', 'precio': 2500, 'stock': 10},
    {'codigo': 'D002', 'nombre': 'Esponja', 'categoria': 'Limpieza', 'precio': 700, 'stock': 40}
]

inventario = pd.DataFrame(datos)
carrito = []
display(inventario)

In [ ]:
# Celda 2. Funciones básicas del carrito
def buscar_productos(inventario, texto, categoria):
    resultado = inventario.copy()
    if texto != '':
        nombres = resultado['nombre'].str.lower()
        resultado = resultado[nombres.str.contains(texto.lower())]
    if categoria != 'Todas':
        resultado = resultado[resultado['categoria'] == categoria]
    return resultado

def agregar_al_carrito(inventario, carrito, codigo, cantidad):
    producto = inventario[inventario['codigo'] == codigo]
    if len(producto) == 0:
        return 'No se encontró el producto.'

    producto = producto.iloc[0]
    if cantidad <= 0:
        return 'La cantidad debe ser mayor que cero.'

    cantidad_actual = 0
    for item in carrito:
        if item['codigo'] == codigo:
            cantidad_actual = item['cantidad']

    if cantidad_actual + cantidad > producto['stock']:
        return 'No hay suficiente stock disponible.'

    if cantidad_actual > 0:
        for item in carrito:
            if item['codigo'] == codigo:
                item['cantidad'] = item['cantidad'] + cantidad
    else:
        carrito.append({
            'codigo': producto['codigo'],
            'nombre': producto['nombre'],
            'precio': producto['precio'],
            'cantidad': cantidad
        })
    return 'Producto agregado al carrito.'

def quitar_del_carrito(carrito, codigo):
    for item in carrito:
        if item['codigo'] == codigo:
            carrito.remove(item)
            return 'Producto eliminado del carrito.'
    return 'El código no está en el carrito.'

def calcular_total(carrito):
    total = 0
    for item in carrito:
        total = total + item['precio'] * item['cantidad']
    return total

def mostrar_carrito(carrito):
    if len(carrito) == 0:
        return 'El carrito está vacío.'
    tabla = pd.DataFrame(carrito)
    tabla['subtotal'] = tabla['precio'] * tabla['cantidad']
    tabla = tabla[['codigo', 'nombre', 'precio', 'cantidad', 'subtotal']]
    return tabla.to_html(index=False) + '<h3>Total: $' + str(calcular_total(carrito)) + '</h3>'

In [ ]:
# Celda 3. Componentes de la interfaz
import ipywidgets as widgets
from IPython.display import display

widget_titulo = widgets.HTML(value='<h1>Carrito de compra - Minimarket</h1><p>Selecciona productos y agrégalos al carrito</p>')
widget_busqueda = widgets.Text(description='Producto:', placeholder='Ej: arroz')
widget_categoria = widgets.Dropdown(description='Categoría:', options=['Todas', 'Abarrotes', 'Bebidas', 'Lácteos', 'Limpieza'], value='Todas')
widget_cantidad = widgets.BoundedIntText(description='Cantidad:', value=1, min=1)
widget_codigo_quitar = widgets.Text(description='Código:', placeholder='Ej: A001')

widget_boton_buscar = widgets.Button(description='Buscar', button_style='info')
widget_boton_agregar = widgets.Button(description='Agregar al carrito', button_style='success')
widget_boton_mostrar = widgets.Button(description='Ver carrito', button_style='info')
widget_boton_quitar = widgets.Button(description='Quitar producto')
widget_boton_vaciar = widgets.Button(description='Vaciar carrito', button_style='danger')
widget_boton_limpiar = widgets.Button(description='Limpiar')

widget_salida = widgets.HTML(value='')
widget_mensaje = widgets.HTML(value='')

In [ ]:
# Celda 4. Conexión de botones con las funciones
def accion_buscar(boton):
    resultados = buscar_productos(inventario, widget_busqueda.value, widget_categoria.value)
    if resultados.empty:
        widget_salida.value = 'No se encontraron productos.'
    else:
        widget_salida.value = resultados.to_html(index=False)

def accion_agregar(boton):
    resultados = buscar_productos(inventario, widget_busqueda.value, widget_categoria.value)
    if len(resultados) == 0:
        widget_mensaje.value = 'Busca un producto antes de agregarlo.'
        return
    codigo = resultados.iloc[0]['codigo']
    widget_mensaje.value = agregar_al_carrito(inventario, carrito, codigo, widget_cantidad.value)
    widget_salida.value = mostrar_carrito(carrito)

def accion_mostrar(boton):
    widget_salida.value = mostrar_carrito(carrito)

def accion_quitar(boton):
    widget_mensaje.value = quitar_del_carrito(carrito, widget_codigo_quitar.value)
    widget_salida.value = mostrar_carrito(carrito)

def accion_vaciar(boton):
    carrito.clear()
    widget_mensaje.value = 'El carrito fue vaciado.'
    widget_salida.value = mostrar_carrito(carrito)

def accion_limpiar(boton):
    widget_busqueda.value = ''
    widget_categoria.value = 'Todas'
    widget_cantidad.value = 1
    widget_codigo_quitar.value = ''
    widget_mensaje.value = ''
    widget_salida.value = ''

widget_boton_buscar.on_click(accion_buscar)
widget_boton_agregar.on_click(accion_agregar)
widget_boton_mostrar.on_click(accion_mostrar)
widget_boton_quitar.on_click(accion_quitar)
widget_boton_vaciar.on_click(accion_vaciar)
widget_boton_limpiar.on_click(accion_limpiar)

In [ ]:
# Celda 5. Organización y visualización
fila_busqueda = widgets.HBox([widget_busqueda, widget_categoria, widget_boton_buscar])
fila_carrito = widgets.HBox([widget_cantidad, widget_boton_agregar, widget_boton_mostrar])
fila_quitar = widgets.HBox([widget_codigo_quitar, widget_boton_quitar, widget_boton_vaciar, widget_boton_limpiar])

widget_formulario = widgets.VBox([
    widgets.HTML('<h3>Buscar productos</h3>'),
    fila_busqueda,
    widgets.HTML('<h3>Carrito de compra</h3>'),
    fila_carrito,
    widgets.HTML('<h3>Modificar carrito</h3>'),
    fila_quitar,
    widget_mensaje
])

widget_app = widgets.VBox([
    widget_titulo,
    widget_formulario,
    widgets.HTML('<h3>Resultados</h3>'),
    widget_salida
])

display(widget_app)